In [17]:
import torch.cuda
from torch.nn import CrossEntropyLoss
from torch.optim import Adam
from dataset import TextDataset
from torch.utils.data import DataLoader
from tqdm import tqdm

def decode_ids(ids, tokenizer):
    tokens = [tokenizer.id_to_token(i) for i in ids]

    text = ""

    for token in tokens:
        if token.startswith("##"):
            text += token[2:]
        else:
            if text:
                text += " "
            text += token

    return text

from model import MiniGPT

with open("corpus.txt", "r") as f:
    text = f.read()

context_window_len = 256
batch_size = 64

dataset = TextDataset(text, context_window_len, context_window_len // 4)

vocab_size = dataset.tokenizer.get_vocab_size()

dataloader = DataLoader(dataset, batch_size=batch_size)

epochs = 20

device = "cuda" if torch.cuda.is_available() else "cpu"

model = MiniGPT(device, vocab_size, context_window_len).to(device)
optimizer = Adam(model.parameters())
criterion = CrossEntropyLoss()

print(decode_ids(model.prompt_one(dataset[0][0].tolist()), dataset.tokenizer))

for epoch in range(epochs):
    total_loss = 0
    model.train()
    for x, y in tqdm(dataloader):

        optimizer.zero_grad()
        x = x.to(device)
        y = y.to(device)
        logits = model(x)

        logits = logits.reshape(-1, logits.shape[-1])
        y = y.reshape(-1)

        loss = criterion(logits, y)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}. Loss = {total_loss / len(dataset)}")
    example = model.prompt_one(dataset[0][0].tolist())
    print(dataset.tokenizer.decode(example))


<BOS> Кривая уточка Русская народная Кривая уточка Жили - были дед да баба . Они пошли в лес за грибами и нашли уточку . А та уточка была кривая . Они ее взяли и принесли домой . Назавтра встали и опять пошли за грибами , а ей сделали утечье гнездышко из перьев . Они ушли , а уточка обернулась девушкой , избу вымыла , воды наносила и пирогов испекла . Дед и баба пришли и спрашивают : — Кто это у нас так все прибрал ? А соседи им говорят : — У вас тут кривенькая девушка воду носила . Вот дед и баба и назавтра ушли , да и спрятались в чулан . Уточка обернулась девушкой и пошла за водой . А дед да баба выскочили да ее перышки и бросили в печь . Перышки все и сгорели . Тут пришла девушка и заплакала . Стала просить у деда , у бабы золотую прялочку . Села на крылечко и прядет куделю . Тут летит стадо гусей - лебедей . Она и говорит : — Гуси мои любезные , дайте мне по перышку . А те говорят : — Другие летят , те дадут . Опять летит стадо гусей - лебедей . — Гуси мои любез ушла своимверху вы